<a href="https://colab.research.google.com/github/TU-USUARIO/labo1-colabs/blob/main/07_Datos_reales_adquisicion_y_derivadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 07 — Datos reales: archivos, series temporales y derivadas numéricas**Laboratorio 1 · Clase 7****Objetivos.**1. Leer archivos de los sistemas de adquisición del laboratorio (Tracker, Pasco/SensorDAQ,   photogate) con encabezados y separadores distintos.2. **Inspeccionar antes de analizar**: recortar el tramo válido de una serie temporal.3. Entender por qué **derivar amplifica el ruido**, y qué hacer al respecto.**Requisitos previos:** Colabs 01 a 06.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy.optimize import curve_fitnp.random.seed(20260923)

---## 1. Los formatos que vas a encontrar- **Tracker** exporta `.txt` o `.csv` con dos o tres líneas de encabezado y columnas `t, x, y`.- **Pasco / SensorDAQ** exportan `.csv` con nombres de columna en la primera fila, a veces con la  unidad entre paréntesis y con coma decimal si la PC está en configuración regional en español.- **Photogate** suele dar un `.txt` con una sola columna de tiempos de paso.La primera regla es **abrir el archivo y mirarlo** antes de leerlo con código. Diez segundos deinspección ahorran media hora de depuración.

In [ ]:
# --- Creamos un archivo de ejemplo con el aspecto de un export de Tracker ---t_ej = np.arange(0, 2.0, 1/60)                     # cámara a 60 fpsx_ej = 0.05 + 1.20*t_ej - 0.5*9.81*t_ej**2x_ej = x_ej + np.random.normal(0, 0.0015, len(t_ej))   # ruido de identificación del puntowith open('tracker_ejemplo.csv', 'w') as f:    f.write("# Tracker 6.1.5 - masa puntual A\n")    f.write("# calibracion: 1.000 m\n")    f.write("t,x\n")    for a, b in zip(t_ej, x_ej):        f.write(f"{a:.5f},{b:.5f}\n")print(open('tracker_ejemplo.csv').read()[:220], "...")

In [ ]:
# Opción A: NumPy. Simple y suficiente casi siempre.datos = np.genfromtxt('tracker_ejemplo.csv', delimiter=',', skip_header=3)t, x = datos[:, 0], datos[:, 1]print("NumPy  ->", t.shape, x.shape)# Opción B: pandas. Mejor cuando hay muchas columnas o nombres útiles.df = pd.read_csv('tracker_ejemplo.csv', comment='#')print("\npandas ->")print(df.head(3))print("\ncolumnas:", list(df.columns))t2, x2 = df['t'].to_numpy(), df['x'].to_numpy()print("¿coinciden ambas lecturas?", np.allclose(t, t2))

Dos problemas frecuentes y su solución:- **Coma decimal:** `pd.read_csv(archivo, decimal=',', sep=';')`.- **Separador desconocido:** `pd.read_csv(archivo, sep=None, engine='python')` lo infiere.

---## 2. Inspeccionar antes de analizarUna serie adquirida casi nunca es aprovechable de punta a punta: hay tramos antes de que empiece elmovimiento, después de que el objeto sale del campo de la cámara, rebotes, o el momento en que se tecruzó la mano. **Recortá el tramo válido a mano, mirando el gráfico, y dejalo documentado en elinforme.**

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))ax.plot(t, x, '.', ms=4)ax.set_xlabel('Tiempo $t$ [s]'); ax.set_ylabel('Posición $x$ [m]')ax.set_title('Serie completa — ¿qué tramo es físicamente válido?')ax.grid(alpha=0.3); fig.tight_layout(); plt.show()print(f"Frecuencia de muestreo: {1/np.mean(np.diff(t)):.1f} Hz")print(f"Duración total        : {t[-1]-t[0]:.3f} s   ({len(t)} puntos)")

In [ ]:
# Recorte con máscara booleana: legible y reversiblemascara = (t >= 0.10) & (t <= 1.60)t_ok, x_ok = t[mascara], x[mascara]print(f"Se conservan {mascara.sum()} de {len(t)} puntos "      f"({100*mascara.sum()/len(t):.0f} %).")

Usá máscaras booleanas y no índices numéricos (`t[12:87]`): si mañana cambiás la frecuencia demuestreo, los índices dejan de significar lo mismo y la máscara sigue funcionando.

---## 3. Derivar amplifica el ruidoPara obtener la velocidad a partir de la posición, la tentación es derivar numéricamente. Se puede—`np.gradient` lo hace bien—, pero hay que entender qué pasa con el error.La diferencia finita entre dos puntos consecutivos es$$ v_i \\approx \\frac{x_{i+1}-x_{i-1}}{2\\Delta t} $$y si cada $x$ tiene incerteza $\\sigma_x$, la velocidad hereda$$ \\sigma_v \\approx \\frac{\\sigma_x}{\\sqrt{2}\\,\\Delta t} $$El $\\Delta t$ está **en el denominador**: cuanto más rápido muestreás, *peor* es la derivada punto apunto. Es contraintuitivo y es la razón por la que muchos informes tienen gráficos de velocidad queparecen ruido puro.

In [ ]:
v_num = np.gradient(x_ok, t_ok)# alternativa: ajustar el modelo y derivar el modelo (analíticamente)def parabola(t, x0, v0, a):    return x0 + v0*t + 0.5*a*t**2pp, _ = curve_fit(parabola, t_ok, x_ok)v_mod = pp[1] + pp[2]*t_okfig, (a1, a2) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)a1.plot(t_ok, x_ok, '.', ms=4, label='datos')a1.plot(t_ok, parabola(t_ok, *pp), 'crimson', lw=1.6, label='ajuste parabólico')a1.set_ylabel('$x$ [m]'); a1.grid(alpha=0.3); a1.legend()a1.set_title('La posición se ve limpia...')a2.plot(t_ok, v_num, '.', ms=4, alpha=0.6, label='np.gradient (derivada numérica)')a2.plot(t_ok, v_mod, 'crimson', lw=1.8, label='derivada del modelo ajustado')a2.set_xlabel('$t$ [s]'); a2.set_ylabel('$v$ [m/s]'); a2.grid(alpha=0.3); a2.legend()a2.set_title('...y la velocidad derivada punto a punto, no')fig.tight_layout(); plt.show()print(f"dispersión de v_num respecto del modelo: {np.std(v_num - v_mod, ddof=1):.4f} m/s")

La conclusión práctica, que vale para todo el curso:> **Cuando tengas un modelo, ajustalo a los datos crudos y derivá el modelo. No derivés los datos.**La derivada numérica sirve para explorar y para detectar tramos raros, no para producir el resultadofinal. Si necesitás la velocidad punto a punto de verdad (por ejemplo, porque no tenés modelo),existen filtros como Savitzky–Golay (`scipy.signal.savgol_filter`), pero cualquier suavizado es unadecisión de análisis que hay que declarar en el informe.

---## 4. Frecuencia de muestreo y submuestreoMuestrear más rápido no siempre es mejor. Tiene dos costos: la derivada empeora (lo acabás de ver) ylos puntos consecutivos dejan de ser independientes si el sensor tiene tiempo de respuesta finito—lo que invalida el supuesto de independencia del Colab 02.Como criterio grueso: para capturar bien un fenómeno periódico se necesitan **al menos 10 puntos porperíodo** (bastante más que el mínimo de 2 que exige el teorema de muestreo, porque no queremos solodetectar la frecuencia sino reconstruir la forma).

In [ ]:
# Submuestreo: quedarse con 1 de cada k puntosfor k in [1, 2, 5]:    tk, xk = t_ok[::k], x_ok[::k]    vk = np.gradient(xk, tk)    pk, _ = curve_fit(parabola, tk, xk)    print(f"1 de cada {k}: {len(tk):3d} puntos | fs = {1/np.mean(np.diff(tk)):5.1f} Hz "          f"| dispersión de v_num = {np.std(vk - (pk[1]+pk[2]*tk), ddof=1):.4f} m/s "          f"| a = {pk[2]:.3f} m/s²")

Fijate en las dos últimas columnas: submuestrear **mejora** mucho la derivada numérica y prácticamente**no cambia** el parámetro obtenido del ajuste. Eso es porque el ajuste ya usa toda la información demanera óptima, mientras que la diferencia finita usa solo dos puntos vecinos.

---## 5. Ejercicios**7.1.** Cargá tu propio archivo de Tracker. Graficá la serie completa, elegí el tramo válido con unamáscara y justificá en una oración dónde cortaste y por qué.**7.2.** Compará las tres adquisiciones de la clase (photogate, Pasco y Tracker) del mismomovimiento. ¿Coinciden las aceleraciones obtenidas? Usá `compatibilidad()` del Colab 03. Si nocoinciden, ¿cuál sospechás que tiene un sistemático?**7.3.** Estimá el error de identificación del punto en Tracker: marcá el mismo cuadro cinco veces ymirá la dispersión. Ése es tu $\\sigma_x$ real, y es el que va en `sigma=` del Colab 06.**7.4.** Con la máscara del ejercicio 7.1, ajustá la parábola con `absolute_sigma=True` usando el$\\sigma_x$ del ejercicio anterior y reportá el $\\chi^2_\\nu$. ¿Te da razonable? Si da mucho mayorque 1 en un tiro de caída libre, la sospecha número uno es que subestimaste $\\sigma_x$.